In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import *
from sklearn.impute import *
from sklearn.preprocessing import *
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import set_config


In [2]:
set_config(transform_output="pandas")

In [ ]:
train_path = 'customer_df_train_dataset-training-master.csv'
test_path = 'customer_df_train_dataset-testing-master.csv'

In [4]:
df_train = pd.read_csv(train_path, index_col='CustomerID')
df_test = pd.read_csv(test_path, index_col='CustomerID')


In [5]:
df_train

,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
CustomerID,,,,,,,,,,,
2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.00,17.0,1.0
3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.00,6.0,1.0
4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.00,3.0,1.0
5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.00,29.0,1.0
6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.00,20.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
449995.0,42.0,Male,54.0,15.0,1.0,3.0,Premium,Annual,716.38,8.0,0.0
449996.0,25.0,Female,8.0,13.0,1.0,20.0,Premium,Annual,745.38,2.0,0.0
449997.0,26.0,Male,35.0,27.0,1.0,5.0,Standard,Quarterly,977.31,9.0,0.0


In [6]:
df_train.columns = [col.lower().replace(' ', '_') for col in df_train.columns]
df_test.columns = [col.lower().replace(' ', '_') for col in df_test.columns]


In [7]:
df_train.contract_length.value_counts()

contract_length
Annual       177198
Quarterly    176530
Monthly       87104
Name: count, dtype: int64

In [8]:
contract_order = ['Annual', 'Quarterly', 'Monthly']
sub_order = ['Premium','Standard', 'Basic']

In [9]:
df_test

,age,gender,tenure,usage_frequency,support_calls,payment_delay,subscription_type,contract_length,total_spend,last_interaction,churn
CustomerID,,,,,,,,,,,
1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
3,47,Male,27,10,2,29,Premium,Annual,757,21,0
4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
5,53,Female,58,24,9,2,Standard,Annual,533,18,0
...,...,...,...,...,...,...,...,...,...,...,...
64370,45,Female,33,12,6,21,Basic,Quarterly,947,14,1
64371,37,Male,6,1,5,22,Standard,Annual,923,9,1
64372,25,Male,39,14,8,30,Premium,Monthly,327,20,1


In [10]:
df_train.dropna(axis='index', inplace=True)
df_test.dropna(axis='index', inplace=True)

In [ ]:
X_train = df_train.drop(columns=['df_train'])
y_train = df_train['df_train']

In [ ]:
X_test = df_test.drop(columns=['df_train'])
y_test = df_test['df_train']

In [13]:
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
cat_features

['gender', 'subscription_type', 'contract_length']

In [14]:
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_features

['age',
 'tenure',
 'usage_frequency',
 'support_calls',
 'payment_delay',
 'total_spend',
 'last_interaction']

#### Building a pipeline for testing different models

In [15]:
def build_preprocessor(scaler_type=None):
    cat_ct = ColumnTransformer([
        ('ohc', OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False), ['gender']),
        ('ordered_encoding', OrdinalEncoder(categories=[sub_order, contract_order]), cat_features[1:])
    ], remainder='passthrough')
    
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('ct', cat_ct)
    ])
    
    num_steps = [('imputer', SimpleImputer(strategy='mean'))]
    if scaler_type == 'standard':
        num_steps.append(('scaler', StandardScaler()))
    elif scaler_type == 'minmax':
        num_steps.append(('scaler', MinMaxScaler()))
    num_pipe = Pipeline(steps=num_steps)
    
    preprocessor = ColumnTransformer([
        ('cat_pipe', cat_pipe, cat_features),
        ('num_pipe', num_pipe, num_features)
    ], remainder='drop')
    
    return preprocessor

base_models = {
    'LogisticRegressionCV': {
        'model': LogisticRegressionCV(Cs=[1.0], random_state=69, max_iter=1000),
        'scaler': 'standard'  
    },
    'RidgeClassifier': {
        'model': RidgeClassifier(alpha=1.0, random_state=69),
        'scaler': 'standard'
    },
    'KNeighborsClassifier': {
        'model': KNeighborsClassifier(n_neighbors=5),
        'scaler': 'standard'   
    },
    'RandomForestClassifier': {
        'model': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=69),
        'scaler': None        
    },
    'AdaBoostClassifier': {
        'model': AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=69),
        'scaler': None
    },
    'XGBClassifier': {
        'model': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.3,
                               subsample=1.0, random_state=69, verbosity=0),
        'scaler': None        
    }
}


In [16]:
def build_pipeline(model_name, params):

    scaler_type = params.get('scaler_type', None)
    
    preprocessor = build_preprocessor(scaler_type)
    
    if model_name == 'LogisticRegressionCV':
        model = LogisticRegressionCV(Cs=[params['lr_Cs']], random_state=69, max_iter=1000)
    elif model_name == 'RidgeClassifier':
        model = RidgeClassifier(alpha=params['ridge_alpha'], random_state=69)
    elif model_name == 'KNeighborsClassifier':
        model = KNeighborsClassifier(n_neighbors=params['knn_n_neighbors'])
    elif model_name == 'RandomForestClassifier':
        model = RandomForestClassifier(
            n_estimators=params['rf_n_estimators'],
            max_depth=params['rf_max_depth'],
            random_state=69
        )
    elif model_name == 'AdaBoostClassifier':
        model = AdaBoostClassifier(
            n_estimators=params['ada_n_estimators'],
            learning_rate=params['ada_lr'],
            random_state=69
        )
    else:  # XGBClassifier
        model = XGBClassifier(
            n_estimators=params['xgb_n_estimators'],
            max_depth=params['xgb_max_depth'],
            learning_rate=params['xgb_learning_rate'],
            subsample=params['xgb_subsample'],
            random_state=69,
            verbosity=0
        )
    
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

Scoring for models

In [17]:
def evaluate_model(model, model_scaler):
    preprocessor = build_preprocessor(model_scaler)
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    scores = cross_val_score(pipeline, X_train, y_train, cv=3,
                             scoring='roc_auc', n_jobs=-1)
    return scores.mean()

baseline_scores = {}
for key, value in base_models.items():
    score = evaluate_model(model=value['model'], model_scaler=value['scaler'])
    baseline_scores[key] = score
    print(f"  {key}: ROC-AUC = {score:.4f}")

sorted_models = sorted(baseline_scores.items(), key=lambda x: x[1], reverse=True)
best_model_names = [name for name, _ in sorted_models[:3]]

  LogisticRegressionCV: ROC-AUC = 0.9448
  RidgeClassifier: ROC-AUC = 0.9448
  KNeighborsClassifier: ROC-AUC = 0.9865
  RandomForestClassifier: ROC-AUC = 1.0000
  AdaBoostClassifier: ROC-AUC = 0.9954
  XGBClassifier: ROC-AUC = 1.0000


In [18]:
def objective(trial, model_name):
    # Собираем словарь параметров для текущего испытания
    params = {}
    
    if model_name in ['LogisticRegressionCV', 'RidgeClassifier', 'KNeighborsClassifier']:
        params['scaler_type'] = trial.suggest_categorical('scaler_type', ['standard', 'minmax'])
    else:
        params['scaler_type'] = None
    
    if model_name == 'LogisticRegressionCV':
        params['lr_Cs'] = trial.suggest_float('lr_Cs', 1e-4, 10.0, log=True)
    elif model_name == 'RidgeClassifier':
        params['ridge_alpha'] = trial.suggest_float('ridge_alpha', 1e-3, 10.0, log=True)
    elif model_name == 'KNeighborsClassifier':
        params['knn_n_neighbors'] = trial.suggest_int('knn_n_neighbors', 3, 21, step=2)
    elif model_name == 'RandomForestClassifier':
        params['rf_n_estimators'] = trial.suggest_int('rf_n_estimators', 50, 500, step=50)
        params['rf_max_depth'] = trial.suggest_int('rf_max_depth', 3, 25)
    elif model_name == 'AdaBoostClassifier':
        params['ada_n_estimators'] = trial.suggest_int('ada_n_estimators', 50, 300, step=50)
        params['ada_lr'] = trial.suggest_float('ada_lr', 0.01, 1.0, log=True)
    else:  # XGBClassifier
        params['xgb_n_estimators'] = trial.suggest_int('xgb_n_estimators', 50, 500, step=50)
        params['xgb_max_depth'] = trial.suggest_int('xgb_max_depth', 3, 25)
        params['xgb_learning_rate'] = trial.suggest_float('xgb_learning_rate', 0.01, 0.5, log=True)
        params['xgb_subsample'] = trial.suggest_float('xgb_subsample', 0.8, 1.0)
    
    pipeline = build_pipeline(model_name, params)
    score = cross_val_score(pipeline, X_train, y_train, cv=3,
                            scoring='roc_auc', n_jobs=-1).mean()
    return score

In [19]:
best_params = {}
for model_name in best_model_names:
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, model_name),
                   n_trials=10, n_jobs=1,   
                   show_progress_bar=True)
    best_params[model_name] = study.best_params

[I 2026-07-01 17:49:05,548] A new study created in memory with name: no-name-26b2f8e9-b0bc-4fdd-bffb-5120f9e9c3a0


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-07-01 17:49:18,336] Trial 0 finished with value: 0.9999994898975052 and parameters: {'xgb_n_estimators': 500, 'xgb_max_depth': 18, 'xgb_learning_rate': 0.04407448137395531, 'xgb_subsample': 0.8577422040434763}. Best is trial 0 with value: 0.9999994898975052.
[I 2026-07-01 17:49:23,822] Trial 1 finished with value: 0.9999993914864594 and parameters: {'xgb_n_estimators': 200, 'xgb_max_depth': 25, 'xgb_learning_rate': 0.10459473272035068, 'xgb_subsample': 0.9496367357642326}. Best is trial 0 with value: 0.9999994898975052.
[I 2026-07-01 17:49:30,856] Trial 2 finished with value: 0.9999946737300901 and parameters: {'xgb_n_estimators': 300, 'xgb_max_depth': 9, 'xgb_learning_rate': 0.018648456292123007, 'xgb_subsample': 0.9518974558589305}. Best is trial 0 with value: 0.9999994898975052.
[I 2026-07-01 17:49:37,314] Trial 3 finished with value: 0.9999992891767011 and parameters: {'xgb_n_estimators': 250, 'xgb_max_depth': 8, 'xgb_learning_rate': 0.40439014098746656, 'xgb_subsample': 0.

[I 2026-07-01 17:50:05,002] A new study created in memory with name: no-name-c0e81a36-f66d-446b-a2a0-de0fc219a03d


[I 2026-07-01 17:50:05,001] Trial 9 finished with value: 0.9999985560930122 and parameters: {'xgb_n_estimators': 350, 'xgb_max_depth': 18, 'xgb_learning_rate': 0.4417113982327251, 'xgb_subsample': 0.8983692296926461}. Best is trial 6 with value: 0.9999995445423607.


...

[I 2026-07-01 17:50:38,839] Trial 0 finished with value: 0.999999508825125 and parameters: {'rf_n_estimators': 200, 'rf_max_depth': 14}. Best is trial 0 with value: 0.999999508825125.
[I 2026-07-01 17:51:47,832] Trial 1 finished with value: 0.9998723989802502 and parameters: {'rf_n_estimators': 500, 'rf_max_depth': 9}. Best is trial 0 with value: 0.999999508825125.
[I 2026-07-01 17:52:24,693] Trial 2 finished with value: 0.999999768906969 and parameters: {'rf_n_estimators': 200, 'rf_max_depth': 22}. Best is trial 2 with value: 0.999999768906969.
[I 2026-07-01 17:52:44,919] Trial 3 finished with value: 0.9974111072184334 and parameters: {'rf_n_estimators': 200, 'rf_max_depth': 5}. Best is trial 2 with value: 0.999999768906969.
[I 2026-07-01 17:53:28,177] Trial 4 finished with value: 0.9997682311082642 and parameters: {'rf_n_estimators': 350, 'rf_max_depth': 8}. Best is trial 2 with value: 0.999999768906969.
[I 2026-07-01 17:54:35,703] Trial 5 finished with value: 0.9999997622099871 and 

[I 2026-07-01 17:55:57,032] A new study created in memory with name: no-name-154f9349-bf4e-47e8-ae11-568177356c05


[I 2026-07-01 17:55:57,030] Trial 9 finished with value: 0.9999995396375292 and parameters: {'rf_n_estimators': 100, 'rf_max_depth': 15}. Best is trial 6 with value: 0.9999997748808024.


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-07-01 17:56:18,426] Trial 0 finished with value: 0.9013090034533033 and parameters: {'ada_n_estimators': 150, 'ada_lr': 0.013989223797994535}. Best is trial 0 with value: 0.9013090034533033.
[I 2026-07-01 17:56:53,244] Trial 1 finished with value: 0.940706057804659 and parameters: {'ada_n_estimators': 250, 'ada_lr': 0.012967619234607156}. Best is trial 1 with value: 0.940706057804659.
[I 2026-07-01 17:57:07,612] Trial 2 finished with value: 0.9013090034533033 and parameters: {'ada_n_estimators': 100, 'ada_lr': 0.026216302354730953}. Best is trial 1 with value: 0.940706057804659.
[I 2026-07-01 17:57:35,348] Trial 3 finished with value: 0.9962510129394717 and parameters: {'ada_n_estimators': 200, 'ada_lr': 0.9251074630037887}. Best is trial 3 with value: 0.9962510129394717.
[I 2026-07-01 17:58:09,928] Trial 4 finished with value: 0.9957421394823683 and parameters: {'ada_n_estimators': 250, 'ada_lr': 0.6125682398888652}. Best is trial 3 with value: 0.9962510129394717.
[I 2026-07-0

In [20]:
final_pipelines = {}
for model_name in best_model_names:
    params = best_params[model_name].copy()  
    
    if model_name in ['RandomForestClassifier', 'XGBClassifier']:
        params.pop('scaler_type', None)
    
    print(f"Обучаем {model_name} с параметрами: {params}")
    pipeline = build_pipeline(model_name, params)
    pipeline.fit(X_train, y_train)
    final_pipelines[model_name] = pipeline

print("\nПредсказания на X_test:")
for name, pipeline in final_pipelines.items():
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    if 'y_test' in locals():
        from sklearn.metrics import roc_auc_score
        auc = roc_auc_score(y_test, y_pred_proba)
        print(f"ROC-AUC на тесте: {auc:.4f}")


Обучаем XGBClassifier с параметрами: {'xgb_n_estimators': 100, 'xgb_max_depth': 7, 'xgb_learning_rate': 0.3012488533278774, 'xgb_subsample': 0.8885904803179524}
Обучаем RandomForestClassifier с параметрами: {'rf_n_estimators': 100, 'rf_max_depth': 24}
Обучаем AdaBoostClassifier с параметрами: {'ada_n_estimators': 200, 'ada_lr': 0.9251074630037887}

Предсказания на X_test:
ROC-AUC на тесте: 0.6709
ROC-AUC на тесте: 0.5591
ROC-AUC на тесте: 0.7794


#### Data drift

So as we can see the accuracy on a test dataset is not so great for some of the best of our models. And why is that? Its because of the data drift in the datasets

In this article https://medium.com/@nursena_kok/from-89-to-58-how-data-drift-destroyed-my-model-and-how-i-caught-it-6f200266c8eb author faced the same issue with this dataset, and she explored this. She used KS test and PSI test and results shown that there are feature distribution drift and a slight target variable drift. PSI showed that some drifts are crucial 

#### The solution

The main solution to this is to train our model on a test (data with new distribution), so it can capture new behaviours of customers

In [26]:
from scipy import stats

feature_cols =  num_features

for col in feature_cols:
    print(f"\n{col}:")
    print(f"  Train → Mean: {df_train[col].mean():.2f}, Std: {df_train[col].std():.2f}")
    print(f"  Test  → Mean: {df_test[col].mean():.2f}, Std: {df_test[col].std():.2f}")
    
    # KS Test
    ks_stat, p_value = stats.ks_2samp(df_train[col], df_test[col])
    
    if p_value < 0.05:
        print(f"p_value {p_value}, ks_stat {ks_stat}")
    else:
        print(f"p_value {p_value}, ks_stat {ks_stat}")


age:
  Train → Mean: 39.37, Std: 12.44
  Test  → Mean: 41.97, Std: 13.92
p_value 0.0, ks_stat 0.1465528858829267

tenure:
  Train → Mean: 31.26, Std: 17.26
  Test  → Mean: 31.99, Std: 17.10
p_value 2.7402015740354127e-33, ks_stat 0.02594840742054355

usage_frequency:
  Train → Mean: 15.81, Std: 8.59
  Test  → Mean: 15.08, Std: 8.82
p_value 8.59427449170323e-87, ks_stat 0.04206249374504001

support_calls:
  Train → Mean: 3.60, Std: 3.07
  Test  → Mean: 5.40, Std: 3.11
p_value 0.0, ks_stat 0.2928200490246105

payment_delay:
  Train → Mean: 12.97, Std: 8.26
  Test  → Mean: 17.13, Std: 8.85
p_value 0.0, ks_stat 0.23935489409246524

total_spend:
  Train → Mean: 631.62, Std: 240.80
  Test  → Mean: 541.02, Std: 260.87
p_value 0.0, ks_stat 0.20193559474730582

last_interaction:
  Train → Mean: 14.48, Std: 8.60
  Test  → Mean: 15.50, Std: 8.64
p_value 7.63732662771742e-224, ks_stat 0.06763267736799994


As we can see from ks_stats there are some serious data drifts...
So it means that df_test features distribtuion changed, and our system fails to predict proper label, being that it was trained on other data...